# Notebook 03: COMPAS Study
**Run after:** Notebook 01

What it does:
1. Uploads and preprocesses compas-scores.csv
2. Prints disparity statistics (DIR, recidivism rates)
3. Runs all five algorithms
4. Computes backdoor ATE at three control levels
5. Saves results to results/compas_results.pkl and results/ate_results.pkl

**Estimated runtime:** ~8-15 minutes (FCI and LiNGAM on n=9380 are the slow steps)

In [ ]:
!pip install -q causal-learn dowhy econml networkx matplotlib numpy pandas scipy scikit-learn seaborn joblib tqdm

In [ ]:
import os, sys
PROJECT_ROOT = '/content/causal_bias_project'   # <-- adjust if needed
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
os.makedirs('results', exist_ok=True)
print('Working directory:', os.getcwd())

## Step 1 — Upload COMPAS dataset
Upload `compas-scores.csv` to the `data/` folder.
The file is available at: https://github.com/propublica/compas-analysis

In [ ]:
# Option A: Upload from your computer
from google.colab import files
os.makedirs('data', exist_ok=True)

# Only upload if file doesn't exist already
if not os.path.exists('data/compas-scores.csv'):
    print('Please upload compas-scores.csv when prompted...')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        with open(f'data/{fname}', 'wb') as f:
            f.write(data)
    print('Uploaded:', list(uploaded.keys()))
else:
    print('compas-scores.csv already present.')

In [ ]:
# Option B (alternative): Download directly from ProPublica GitHub
# Uncomment and run this cell instead of Option A
# !wget -q -O data/compas-scores.csv \
#   'https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv'
# print('Downloaded compas-scores.csv')

## Step 2 — Preprocess and run causal discovery

In [ ]:
import importlib
import experiments.run_compas as rc
importlib.reload(rc)

compas_results = rc.main('data/compas-scores.csv')
print('\nAlgorithms completed:', [k for k in compas_results if not k.startswith('_')])

## Step 3 — Backdoor ATE estimation

In [ ]:
import experiments.dowhy_ate as ate_mod
importlib.reload(ate_mod)

ate_results = ate_mod.main()
print('\nATE results:')
for k, v in ate_results.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:.4f}')

## Step 4 — Cross-algorithm comparison table

In [ ]:
import pickle, pandas as pd

with open('results/compas_results.pkl', 'rb') as f:
    cr = pickle.load(f)

rows = []
for alg in ['PC','FCI','GES','GRaSP','ICA-LiNGAM','DirectLiNGAM']:
    if alg in cr:
        r = cr[alg]
        b = r.get('beta_hat')
        rows.append({
            'Algorithm':     alg,
            'Race→Score':    '✓' if r.get('race_score') else '✗',
            'Race→Priors':   '✓' if r.get('race_prior') else '✗',
            'Latent conf.':  '↔' if r.get('n_bidirected',0) > 0 else 'No',
            'β̂':            f'{b:+.4f}' if b is not None else '—',
        })

df_compare = pd.DataFrame(rows)
print('\nCOMPAS Algorithm Comparison:')
print(df_compare.to_string(index=False))

print(f'\nBackdoor ATE (full controls): {ate_results["ate_full"]:+.4f}')
print(f'LiNGAM β̂ Race→Score:         {cr["ICA-LiNGAM"]["beta_hat"]:+.4f}')
print(f'Difference:                   {abs(ate_results["ate_full"] - cr["ICA-LiNGAM"]["beta_hat"]):.4f}')

## Step 5 — Generate Figures 4, 5, 6

In [ ]:
os.makedirs('figures/output', exist_ok=True)

import figures.fig_compas_dag as f4
importlib.reload(f4)
f4.make_figure()

import figures.fig_directlingam_compas as f5
importlib.reload(f5)
f5.make_figure()

import figures.fig_corr_vs_causal as f6
importlib.reload(f6)
f6.make_figure()

print('Figures generated:', os.listdir('figures/output/'))

In [ ]:
from IPython.display import Image, display
for fname in ['fig_compas_dag.png','fig_directlingam_compas.png','fig_corr_vs_causal.png']:
    p = f'figures/output/{fname}'
    if os.path.exists(p):
        print(f'── {fname}')
        display(Image(p))